# Tool use as actions — interactive companion

Companion to [Post 3b: Tool Use as Actions](../posts/03b-tool-use.qmd).

When an LM can call tools, it becomes an agent acting in an extended MDP.
This notebook lets you build and run the ReAct loop, watch tool use beat
internal computation, and see errors compound over multi-hop chains.

**You'll do (~20 minutes):**
1. See the internal-accuracy cliff vs the exact calculator.
2. Run a ReAct episode on a multi-hop knowledge-graph task.
3. Find the optimal tool-use threshold from expected utility.
4. Watch errors compound as $(1-p)^L$ over chain length.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.tools import (
    ArithmeticProblem, NoisyArithmeticModel, sample_problem,
    make_calculator, make_lookup, KnowledgeGraph,
    build_example_graph, run_react_oracle, run_react_noisy,
)

rng = np.random.default_rng(0)

## 1. The internal-accuracy cliff

We model an agent whose internal arithmetic degrades with problem size —
exactly the failure mode that motivates giving an LLM a calculator.

In [ ]:
model = NoisyArithmeticModel(base_accuracy=0.99, decay=0.45, d0=2.0)
calc = make_calculator()

# Measure internal vs tool accuracy by operand size.
digit_levels = [1, 2, 3, 4, 5]
internal_acc, tool_acc = [], []
for md_ in digit_levels:
    r = np.random.default_rng(md_)
    problems = []
    for _ in range(800):
        a = int(r.integers(10**(md_-1), 10**md_))
        b = int(r.integers(10**(md_-1), 10**md_))
        op = str(r.choice(["+", "-", "*"]))
        problems.append(ArithmeticProblem(a, op, b))
    sr = np.random.default_rng(1)
    internal_acc.append(np.mean([model.solve_internally(p, sr)[1] for p in problems]))
    tool_acc.append(np.mean([calc(p.as_tuple())[0] == p.answer for p in problems]))

plt.plot(digit_levels, internal_acc, "o-", color="#c44e52", lw=2, ms=10,
         label="internal computation")
plt.plot(digit_levels, tool_acc, "s-", color="#3a7ebf", lw=2, ms=10,
         label="calculator tool")
plt.xlabel("operand size (digits)"); plt.ylabel("accuracy")
plt.title("The internal-accuracy cliff"); plt.legend()
plt.grid(alpha=0.3); plt.ylim(-0.05, 1.05); plt.show()

Internal accuracy falls off a cliff; the calculator stays flat at 100%.
This gap is the entire motivation for tool use.

### Try this
- Change `decay=0.45` to `decay=0.2`. The cliff is gentler — the model is
  more capable internally. Does the tool still help on big problems?
- Change `decay` to `0.9`. Now even 2-digit problems are unreliable internally.

## 2. A ReAct episode on a multi-hop task

A knowledge graph where answering requires chaining lookups. Watch the
think-act-observe loop build up the answer one observation at a time.

In [ ]:
kg, task = build_example_graph(np.random.default_rng(0), chain_len=3, seed=3)
lookup = make_lookup(kg)
trace = run_react_oracle(kg, task, lookup)

print(f"Task: start at {task.start}, follow chain {task.chain}")
print(f"True answer: {task.answer}\n")
print("ReAct trace:")
for kind, content in trace.steps:
    if kind == "act":
        tool, args = content
        print(f"  [ACT]     call {tool}{args}")
    elif kind == "observe":
        print(f"  [OBSERVE] -> {content}")
    elif kind == "answer":
        print(f"  [ANSWER]  {content}")
    else:
        print(f"  [THINK]   {content}")
print(f"\nCorrect: {trace.correct}  ({trace.n_tool_calls} tool calls)")

The agent threads each observation into the next lookup — it can't shortcut,
because the intermediate entities are hidden until queried. This is the
POMDP belief update: each observation collapses uncertainty about the next hop.

### Try this
- Change `chain_len=3` to `chain_len=5`. More hops, more tool calls.
- Change the `seed` to get a different graph and chain.

## 3. When should the agent call the tool?

Tools cost something. The rational rule: call the tool when internal
accuracy drops below `1 - cost`. Let's verify by sweeping the threshold.

In [ ]:
# Sample a mix of problems.
rng = np.random.default_rng(0)
problems = [sample_problem(rng, max_digits=4) for _ in range(3000)]
calc = make_calculator()

lam = 0.3  # tool cost (try changing this!)
thresholds = np.linspace(0, 16, 40)
utilities = []
solve_rng = np.random.default_rng(1)
for thr in thresholds:
    u = 0.0
    for p in problems:
        if p.difficulty >= thr:
            u += (1.0 if calc(p.as_tuple())[0] == p.answer else 0.0) - lam
        else:
            u += 1.0 if model.solve_internally(p, solve_rng)[1] else 0.0
    utilities.append(u / len(problems))

best_thr = thresholds[int(np.argmax(utilities))]
# Analytical prediction: P(correct) = 1 - lam.
analytical = model.d0 - np.log((1 - lam) / model.base_accuracy) / model.decay

plt.plot(thresholds, utilities, "-", color="#3a7ebf", lw=2)
plt.axvline(best_thr, color="#55a467", ls="-", lw=2,
            label=f"empirical best = {best_thr:.1f}")
plt.axvline(analytical, color="#c44e52", ls="--", lw=2,
            label=f"analytical = {analytical:.1f}")
plt.xlabel("tool-use threshold"); plt.ylabel(f"net utility (cost λ={lam})")
plt.title("Optimal tool-use threshold"); plt.legend()
plt.grid(alpha=0.3); plt.show()

The empirical optimum matches the analytical rule "call the tool when
internal accuracy < 1 − λ."

### Try this
- Set `lam = 0.05` (cheap tools). The threshold drops — use tools more.
- Set `lam = 0.7` (expensive tools). The threshold rises — reserve tools for
  the hardest problems only.

## 4. Errors compound over chains

The central challenge of long-horizon agents. A noisy agent picks the wrong
relation with probability p at each step. Watch success decay with chain length.

In [ ]:
chain_lengths = list(range(1, 11))
error_rates = [0.0, 0.05, 0.1, 0.2]
colors = plt.cm.plasma(np.linspace(0, 0.8, len(error_rates)))

for p_err, c in zip(error_rates, colors):
    success = []
    for cl in chain_lengths:
        n_correct = 0
        for t in range(300):
            kg, task = build_example_graph(np.random.default_rng(t),
                                            chain_len=cl, seed=t)
            lookup = make_lookup(kg)
            trace = run_react_noisy(kg, task, lookup,
                                     rng=np.random.default_rng(t + 5000),
                                     wrong_relation_prob=p_err, max_steps=20)
            n_correct += int(trace.correct)
        success.append(n_correct / 300)
    plt.plot(chain_lengths, success, "o-", color=c, lw=2,
             label=f"per-step error = {p_err}")
    # (1-p)^L lower bound.
    plt.plot(chain_lengths, [(1-p_err)**cl for cl in chain_lengths],
             "--", color=c, alpha=0.5)

plt.xlabel("chain length L"); plt.ylabel("success rate")
plt.title("Error compounding: solid = empirical, dashed = $(1-p)^L$ bound")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(-0.05, 1.05); plt.show()

Even a 10% per-step error rate collapses to ~55% success on a 10-hop chain.
Note the empirical curves sit *above* the $(1-p)^L$ dashed lines — in a small
graph, a wrong relation sometimes lands on the right answer by luck, so the
clean exponential is a lower bound.

This exponential sensitivity to per-step reliability is *the* reason
long-horizon agents are hard, and it motivates the search-based recovery
of the next post.

### Try this
- Push `error_rates` to include 0.4. How fast does that collapse?
- Increase the per-chain-length task count from 300 for smoother curves.

## What's next

You've built the agent loop from scratch:

- **Tool use as an extended MDP** — actions augmented with tool calls,
  observations from the environment.
- **ReAct** — the think-act-observe cycle, threading observations into
  subsequent actions.
- **When to call a tool** — an expected-utility threshold.
- **Error compounding** — the exponential that makes long horizons hard.

The next post — [Post 3c: Tree of Thoughts and MCTS](../posts/03c-tree-of-thoughts.qmd) —
attacks the error-compounding problem with search. Instead of committing to
one ReAct path, the agent explores a tree of reasoning branches, expanding
promising ones and pruning dead ends. It's this post's action-space view
combined with the tree search behind AlphaGo.